In [20]:
"""
convert_5min_to_15min.py
------------------------
Converts 5-minute candle JSON data to 15-minute timeframe and saves as CSV.

Input JSON schema (list of candles):
  {
    "symbol":       str,
    "candle_open":  "HH:MM:SS",
    "candle_close": "HH:MM:SS",
    "timeframe":    "5min",
    "ohlcv": {
      "open": float, "high": float, "low": float,
      "close": float, "volume": int
    },
    "aggression": {
      "score": float,
      "buy_volume": int,
      "sell_volume": int,
      "buy_percentage": float,   # optional
      "sell_percentage": float   # optional
    },
    "limit_orders": {
      "total_bid_qty": int,
      "total_ask_qty": int,
      "qty_ratio": float,
      "total_bid_orders": int,   # optional
      "total_ask_orders": int,   # optional
      "order_ratio": float,
      "avg_bid_order_size": float,  # optional
      "avg_ask_order_size": float,  # optional
      "order_size_ratio": float
    },
    "signal": str   # optional — "1", "-1", "0"
  }

Usage:
  python convert_5min_to_15min.py "D:\\FM\\candle_maker\\artifacts\\candle_data - 30-04-2026.json"
  python convert_5min_to_15min.py input.json --output result.csv
  python convert_5min_to_15min.py input.json --output result.csv --session-start 09:15
"""

import json
import csv
import argparse
import sys
from pathlib import Path
from collections import defaultdict


# Notebook override paths (edit these two variables if you want)
input_path = r"D:\FM\candle_maker\artifacts\candle_data - 29-04-2026.json"  # e.g. r"artifacts\candle_data.json"
output_path = r"D:\FM\candle_maker\artifacts\candle_data_30min_29_04.csv"  # e.g. r"artifacts\candle_data_15min.csv"

# Aggregation timeframe in minutes (e.g. 15, 30, 60)
tf = 30


def _candidate_artifact_dirs() -> list[Path]:
    """Return likely artifact directories regardless of notebook CWD."""
    cwd = Path.cwd().resolve()
    roots = [cwd, *cwd.parents]
    dirs = []
    for root in roots:
        candidate = root / "artifacts"
        if candidate.exists() and candidate.is_dir():
            dirs.append(candidate)
    return dirs


def time_to_min(t: str) -> int:
    """'HH:MM:SS' or 'HH:MM' -> total minutes since midnight."""
    parts = t.split(":")
    return int(parts[0]) * 60 + int(parts[1])


def min_to_time(m: int) -> str:
    """Total minutes -> 'HH:MM:SS'."""
    return f"{m // 60:02d}:{m % 60:02d}:00"


def min_to_close_time(m: int) -> str:
    """Total minutes -> close time string 'HH:MM:59'."""
    return f"{m // 60:02d}:{m % 60:02d}:59"


def aggregate_candles(candles: list, session_start_min: int, timeframe_minutes: int) -> dict:
    """
    Aggregate a list of 5-min candles into a single 15-min bar.

    OHLCV:
      open   = first candle open
      high   = max of all highs
      low    = min of all lows
      close  = last candle close
      volume = sum

    Aggression:
      buy_volume / sell_volume = sum
      score, buy_percentage, sell_percentage = recomputed from totals

    Limit orders:
      total_bid_qty / total_ask_qty = sum  ->  qty_ratio recomputed
      order_ratio / order_size_ratio = averaged (point-in-time snapshots)

    Signal (optional):
      +1 wins if any candle = 1 and none = -1
      -1 wins if any candle = -1 and none = +1
      conflict or all-zero -> "0"
    """
    candles = sorted(candles, key=lambda c: c["candle_open"])
    first, last = candles[0], candles[-1]
    n = len(candles)

    # --- OHLCV ---
    open_  = first["ohlcv"]["open"]
    high   = max(c["ohlcv"]["high"] for c in candles)
    low    = min(c["ohlcv"]["low"]  for c in candles)
    close  = last["ohlcv"]["close"]
    volume = sum(c["ohlcv"]["volume"] for c in candles)

    # --- Aggression ---
    buy_vol  = sum(c["aggression"]["buy_volume"]  for c in candles)
    sell_vol = sum(c["aggression"]["sell_volume"] for c in candles)
    total_vol = buy_vol + sell_vol
    if total_vol > 0:
        score      = round((buy_vol - sell_vol) / total_vol, 4)
        buy_pct    = round(buy_vol  / total_vol * 100, 4)
        sell_pct   = round(sell_vol / total_vol * 100, 4)
    else:
        score = buy_pct = sell_pct = 0.0

    # --- Limit orders ---
    bid_qty = sum(c["limit_orders"]["total_bid_qty"] for c in candles)
    ask_qty = sum(c["limit_orders"]["total_ask_qty"] for c in candles)
    qty_ratio = round(bid_qty / ask_qty, 4) if ask_qty > 0 else 0.0

    # Average the ratio fields (point-in-time snapshots, not cumulative)
    avg_order_ratio     = round(sum(c["limit_orders"]["order_ratio"]      for c in candles) / n, 4)
    avg_order_size_ratio= round(sum(c["limit_orders"]["order_size_ratio"] for c in candles) / n, 4)

    # Optional fields that exist in some schemas
    lo_out = {
        "total_bid_qty":       bid_qty,
        "total_ask_qty":       ask_qty,
        "qty_ratio":           qty_ratio,
        "avg_order_ratio":     avg_order_ratio,
        "avg_order_size_ratio":avg_order_size_ratio,
    }
    # Preserve total_bid_orders / total_ask_orders if present
    if "total_bid_orders" in first["limit_orders"]:
        lo_out["total_bid_orders"] = sum(c["limit_orders"].get("total_bid_orders", 0) for c in candles)
        lo_out["total_ask_orders"] = sum(c["limit_orders"].get("total_ask_orders", 0) for c in candles)

    # --- Signal (optional) ---
    signal = None
    if "signal" in first:
        sigs = [int(c.get("signal", 0)) for c in candles]
        has_pos = any(s == 1  for s in sigs)
        has_neg = any(s == -1 for s in sigs)
        if has_pos and not has_neg:
            signal = "1"
        elif has_neg and not has_pos:
            signal = "-1"
        else:
            signal = "0"

    # --- Window timestamps ---
    open_min  = time_to_min(first["candle_open"])
    win_idx   = (open_min - session_start_min) // timeframe_minutes
    win_start = session_start_min + win_idx * timeframe_minutes
    win_end   = win_start + (timeframe_minutes - 1)  # last minute of the window

    bar = {
        "symbol":       first["symbol"],
        "candle_open":  min_to_time(win_start),
        "candle_close": min_to_close_time(win_end),
        "timeframe":    f"{timeframe_minutes}min",
        "ohlcv": {
            "open":   open_,
            "high":   high,
            "low":    low,
            "close":  close,
            "volume": volume,
        },
        "aggression": {
            "score":           score,
            "buy_volume":      buy_vol,
            "sell_volume":     sell_vol,
            "buy_percentage":  buy_pct,
            "sell_percentage": sell_pct,
        },
        "limit_orders": lo_out,
        "candles_count": n,
    }

    if signal is not None:
        bar["signal"] = signal

    return bar


def convert(data: list, session_start: str = "09:15", timeframe_minutes: int = 15) -> list:
    """
    Group 5-min candles by (symbol, fixed-minute window) and aggregate each group.

    Args:
        data:          List of 5-min candle dicts.
        session_start: Market session start time 'HH:MM' (default '09:15').
                       Used to align window boundaries correctly.
        timeframe_minutes: Aggregation window in minutes (e.g. 15, 30, 60).

    Returns:
        List of aggregated bar dicts, sorted by (symbol, candle_open).
    """
    base = time_to_min(session_start)
    groups = defaultdict(list)

    for candle in data:
        t = time_to_min(candle["candle_open"])
        if t < base:
            # Skip pre-session candles
            continue
        win_idx   = (t - base) // timeframe_minutes
        win_start = base + win_idx * timeframe_minutes
        key = (candle["symbol"], win_start)
        groups[key].append(candle)

    bars = [
        aggregate_candles(candles, base, timeframe_minutes=timeframe_minutes)
        for candles in groups.values()
    ]

    bars.sort(key=lambda b: (b["symbol"], b["candle_open"]))
    return bars


CSV_COLUMNS = [
    "symbol", "candle_open", "candle_close", "timeframe",
    "open", "high", "low", "close", "volume",
    "aggr_score", "buy_volume", "sell_volume", "buy_pct", "sell_pct",
    "total_bid_qty", "total_ask_qty", "qty_ratio",
    "total_bid_orders", "total_ask_orders",
    "avg_order_ratio", "avg_order_size_ratio",
    "signal", "candles_count",
]


def bars_to_csv(bars: list, output_path: Path) -> None:
    """Write list of 15-min bar dicts to a flat CSV file."""
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
        writer.writeheader()
        for b in bars:
            row = {
                "symbol":               b["symbol"],
                "candle_open":          b["candle_open"],
                "candle_close":         b["candle_close"],
                "timeframe":            b["timeframe"],
                "open":                 b["ohlcv"]["open"],
                "high":                 b["ohlcv"]["high"],
                "low":                  b["ohlcv"]["low"],
                "close":                b["ohlcv"]["close"],
                "volume":               b["ohlcv"]["volume"],
                "aggr_score":           b["aggression"]["score"],
                "buy_volume":           b["aggression"]["buy_volume"],
                "sell_volume":          b["aggression"]["sell_volume"],
                "buy_pct":              b["aggression"]["buy_percentage"],
                "sell_pct":             b["aggression"]["sell_percentage"],
                "total_bid_qty":        b["limit_orders"]["total_bid_qty"],
                "total_ask_qty":        b["limit_orders"]["total_ask_qty"],
                "qty_ratio":            b["limit_orders"]["qty_ratio"],
                "total_bid_orders":     b["limit_orders"].get("total_bid_orders", ""),
                "total_ask_orders":     b["limit_orders"].get("total_ask_orders", ""),
                "avg_order_ratio":      b["limit_orders"]["avg_order_ratio"],
                "avg_order_size_ratio": b["limit_orders"]["avg_order_size_ratio"],
                "signal":               b.get("signal", ""),
                "candles_count":        b["candles_count"],
            }
            writer.writerow(row)


def main():
    parser = argparse.ArgumentParser(
        description="Convert 5-min candle JSON to 15-min timeframe CSV."
    )
    parser.add_argument(
        "input",
        nargs="?",
        default=None,
        help='Path to input 5-min JSON file. '
             'Example: "D:\\FM\\candle_maker\\artifacts\\candle_data - 30-04-2026.json"'
    )
    parser.add_argument(
        "--output", "-o",
        default=None,
        help="Output CSV file path. Defaults to <input_stem>_15min.csv in the same folder."
    )
    parser.add_argument(
        "--session-start", "-s",
        default="09:15",
        help="Market session start time HH:MM (default: 09:15)."
    )

    # parse_known_args avoids notebook kernel flags breaking argparse
    args, _ = parser.parse_known_args()

    # Notebook users can override paths by editing the module-level variables
    # `input_path` and `output_path` (above). If left as None, argparse + fallback apply.
    override_input_path = globals().get("input_path", None)
    override_output_path = globals().get("output_path", None)

    input_path = (
        Path(override_input_path).expanduser()
        if override_input_path
        else (Path(args.input).expanduser() if args.input else None)
    )

    # If input is relative, resolve from current working directory.
    if input_path is not None and not input_path.is_absolute():
        input_path = (Path.cwd() / input_path).resolve()

    if input_path is None and "ipykernel" in Path(sys.argv[0]).name.lower():
        candidates = []
        searched_dirs = _candidate_artifact_dirs()
        for artifacts_dir in searched_dirs:
            candidates.extend(artifacts_dir.glob("candle_data*.json"))

        candidates = sorted({p.resolve() for p in candidates}, key=lambda p: p.stat().st_mtime)
        if candidates:
            input_path = candidates[-1]
            print(f"No input provided. Using latest artifact file: {input_path}")
        else:
            searched = "\n".join(str(d) for d in searched_dirs) or str(Path.cwd())
            raise ValueError(
                "No input path provided and no matching candle_data*.json found.\n"
                f"Searched artifact directories:\n{searched}\n"
                "Pass an input path explicitly, e.g.:\n"
                "python convert_5min_to_15min.py \"artifacts\\candle_data.json\""
            )

    if input_path is None:
        raise ValueError("Input file path is required.")

    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    if override_output_path:
        output_path = Path(override_output_path).expanduser()
        if not output_path.is_absolute():
            output_path = (Path.cwd() / output_path).resolve()
    else:
        output_path = Path(args.output) if args.output else \
            input_path.parent / (input_path.stem + "_15min.csv")

    print(f"Reading : {input_path}")
    with open(input_path, encoding="utf-8") as f:
        data = json.load(f)

    print(f"Input   : {len(data)} x 5-min candles")

    # use global `tf` as timeframe in minutes
    bars = convert(data, session_start=args.session_start, timeframe_minutes=tf)

    symbols = sorted(set(b["symbol"] for b in bars))
    print(f"Output  : {len(bars)} 15-min bars across {len(symbols)} symbol(s)")
    for sym in symbols:
        sym_bars  = [b for b in bars if b["symbol"] == sym]
        incomplete = [b for b in sym_bars if b["candles_count"] < 3]
        print(f"  {sym}: {len(sym_bars)} bars", end="")
        if incomplete:
            times = [b["candle_open"] for b in incomplete]
            print(f"  [⚠ {len(incomplete)} incomplete bar(s): {times}]", end="")
        print()

    bars_to_csv(bars, output_path)
    print(f"Saved   : {output_path}")


if __name__ == "__main__":
    main()

Reading : D:\FM\candle_maker\artifacts\candle_data - 29-04-2026.json
Input   : 134 x 5-min candles
Output  : 24 15-min bars across 2 symbol(s)
  NSE:ASIANPAINT-EQ: 12 bars  [⚠ 1 incomplete bar(s): ['14:45:00']]
  NSE:HEROMOTOCO-EQ: 12 bars  [⚠ 1 incomplete bar(s): ['14:45:00']]
Saved   : D:\FM\candle_maker\artifacts\candle_data_30min_29_04.csv
